In [4]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q 'rfdetr[train,loggers]' albumentations opencv-python-headless ensemble-boxes scikit-learn
print('✅ 설치 완료')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 설치 완료


In [5]:
import os, json, glob, shutil, random, subprocess, re
import numpy as np
from collections import defaultdict, Counter
from sklearn.model_selection import StratifiedGroupKFold
import torch

# ── AI Hub 설정 ──────────────────────────────────────────────
AIHUB_API_KEY = ''  # ← API 키 입력
DATASET_KEY   = '576'

# ── 금지 파일 ────────────────────────────────────────────────
FORBIDDEN = ['TL_2_조합', 'TS_2_조합']

# ── 경로 ─────────────────────────────────────────────────────
SHARED_ID   = '1SlNpnNKD7cPbwLJH0A9-WUcx1Fhbzfrd'
SHARED_DIR  = f'/content/drive/.shortcut-targets-by-id/{SHARED_ID}/ai12-level1-project'
TRAIN_ANN   = f'{SHARED_DIR}/sprint_ai_project1_data/train_annotations'
TRAIN_IMG   = f'{SHARED_DIR}/sprint_ai_project1_data/train_images'
MY_OUTPUT   = '/content/drive/MyDrive/pill_detection_outputs'
AIHUB_DIR   = '/content/aihub_pills'
DATASET_DIR = '/content/dataset_v2'
AUG_DIR     = '/content/dataset_v2_aug'
EXP_NAME    = 'final_v2_aihub'
SEED, N_SPLITS, N_RANDOM = 42, 5, 15

for d in [MY_OUTPUT, AIHUB_DIR, DATASET_DIR, AUG_DIR]:
    os.makedirs(d, exist_ok=True)

# ── 13개 필수 약품 ────────────────────────────────────────────
TARGET_PILLS = [
    '세비카정', '비타비백정', '조인스정', '낙소졸정',
    '란스톤엘에프디티정', '쎄로켈정', '펠루비정', '게루삼엠정',
    '케이캡정', '쿠에타핀정', '넥시움정', '브린텔릭스정', '렉사프로정',
    # ← 14번째: 섹션 5 (ImageSearch) 실행 후 여기에 추가
]
UNKNOWN_PILL_14 = None  # ImageSearch로 확인 후 약품명 입력

if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f'GPU: {torch.cuda.get_device_name(0)} ({vram:.1f}GB)')
    BATCH_SIZE = 8 if vram >= 16 else 4
    GRAD_ACCUM = 2 if vram >= 16 else 4
else:
    raise RuntimeError('GPU 필요')

print(f'필수 약품: {len(TARGET_PILLS)}개 (14번째 미확정)')
print(f'랜덤 추가: {N_RANDOM}개')


GPU: NVIDIA A100-SXM4-40GB (42.4GB)
필수 약품: 13개 (14번째 미확정)
랜덤 추가: 15개


## 1. aihubshell 설치 (선택 — 파일 목록 조회용)

> Colab은 해외 IP라 실제 다운로드는 불가합니다.
> 파일 목록 조회 목적으로만 사용합니다.


In [6]:
subprocess.run(['curl', '-o', '/content/aihubshell',
                'https://api.aihub.or.kr/api/aihubshell.do', '-s'])
subprocess.run(['chmod', '+x', '/content/aihubshell'])
print('✅ aihubshell 설치 완료')


✅ aihubshell 설치 완료


## 2. AI Hub 파일 목록 조회


In [7]:
print('파일 목록 조회 중...')
result = subprocess.run(
    ['/content/aihubshell', '-mode', 'l',
     '-datasetkey', DATASET_KEY,
     '-aihubapikey', AIHUB_API_KEY],
    capture_output=True, text=True, encoding='utf-8'
)
FILE_LIST_RAW = result.stdout
with open('/content/aihub_filelist.txt', 'w', encoding='utf-8') as f:
    f.write(FILE_LIST_RAW)

lines = [l.strip() for l in FILE_LIST_RAW.split('\n') if l.strip()]
print(f'전체 라인: {len(lines)}개')

# 파일 구조 파싱
file_info = []  # [(key, name, size, type)]
for line in lines:
    if '|' not in line: continue
    if any(fb in line for fb in FORBIDDEN): continue
    key = re.findall(r'\|\s*(\d{5,})\s*$', line.strip())
    if not key: continue
    name = line.split('|')[0].strip().lstrip('│├└─').strip()
    size = line.split('|')[1].strip() if len(line.split('|'))>1 else ''
    ftype = ('label_train' if 'TL_' in name else
             'label_val'   if 'VL_' in name else
             'img_train'   if 'TS_' in name else
             'img_val'     if 'VS_' in name else 'other')
    file_info.append({'key':key[0], 'name':name, 'size':size, 'type':ftype})

import pandas as pd
df_files = pd.DataFrame(file_info)
print('\n파일 유형별 수:')
print(df_files['type'].value_counts())
display(df_files)


파일 목록 조회 중...
전체 라인: 225개

파일 유형별 수:
type
label_train    88
img_train      88
img_val        11
label_val      11
Name: count, dtype: int64


,key,name,size,type
0,66065,│ │ ├─TL_1_조합.zip,8 MB,label_train
1,66067,│ │ ├─TL_3_조합.zip,7 MB,label_train
2,66068,│ │ ├─TL_4_조합.zip,7 MB,label_train
3,66069,│ │ ├─TL_5_조합.zip,8 MB,label_train
4,66070,│ │ ├─TL_6_조합.zip,8 MB,label_train
...,...,...,...,...
193,66249,VL_5_단일.zip,17 MB,label_val
194,66250,VL_6_단일.zip,17 MB,label_val
195,66251,VL_7_단일.zip,15 MB,label_val
196,66252,VL_8_단일.zip,17 MB,label_val


## 3. AI Hub 데이터 다운로드 (Google Drive에서)

AI Hub는 해외(Colab) IP에서 직접 다운로드가 차단됩니다.
로컬 PC에서 미리 다운로드해 Drive에 저장해둔 파일을 사용합니다.

```
VL_image.tar       → Validation 이미지
VL_image.jason.tar → Validation 라벨 JSON
```


In [8]:
!pip install gdown -q
import gdown

# ── Drive 파일 ID ─────────────────────────────────────────────
IMG_FILE_ID  = '1R0vx41HeHiUqpML6iFI0gFszz-f-z7uW'  # VL_image.tar
JSON_FILE_ID = '1Taden_B2LklBEHx4BY4Gl9CzytcpQt-G'  # VL_image.jason.tar

IMG_TAR  = '/content/VL_image.tar'
JSON_TAR = '/content/VL_image_json.tar'

# ── Drive → 로컬 다운로드 ────────────────────────────────────
if not os.path.exists(IMG_TAR):
    print('이미지 tar 다운로드 중...')
    gdown.download(f'https://drive.google.com/uc?id={IMG_FILE_ID}',
                   IMG_TAR, quiet=False)
else:
    print('이미지 tar 이미 존재')

if not os.path.exists(JSON_TAR):
    print('JSON tar 다운로드 중...')
    gdown.download(f'https://drive.google.com/uc?id={JSON_FILE_ID}',
                   JSON_TAR, quiet=False)
else:
    print('JSON tar 이미 존재')

# ── 압축 해제 ─────────────────────────────────────────────────
print('\n압축 해제 중...')
os.makedirs(AIHUB_DIR, exist_ok=True)
!tar -xf {IMG_TAR}  -C {AIHUB_DIR}
!tar -xf {JSON_TAR} -C {AIHUB_DIR}

# ── 결과 확인 ────────────────────────────────────────────────
label_jsons = glob.glob(os.path.join(AIHUB_DIR,'**','*.json'), recursive=True)
label_jsons = [j for j in label_jsons if not any(fb in j for fb in FORBIDDEN)]
aihub_imgs  = glob.glob(os.path.join(AIHUB_DIR,'**','*.png'), recursive=True)
aihub_imgs += glob.glob(os.path.join(AIHUB_DIR,'**','*.jpg'), recursive=True)

print(f'\n✅ 완료')
print(f'  이미지: {len(aihub_imgs)}장')
print(f'  JSON  : {len(label_jsons)}개')

# 폴더 구조 미리보기
print('\n폴더 구조:')
for root, dirs, files in os.walk(AIHUB_DIR):
    depth = root.replace(AIHUB_DIR,'').count(os.sep)
    if depth > 2: continue
    print('  '*depth + os.path.basename(root) + '/')
    for f in sorted(files)[:3]:
        size = os.path.getsize(os.path.join(root,f))/1e6
        print('  '*(depth+1) + f + f' ({size:.1f}MB)')


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
      K-018357-020238-024752_0_2_0_2_90_000_200.json (0.0MB)
    K-024752/
      K-018357-020238-024752_0_2_0_2_70_000_200.json (0.0MB)
      K-018357-020238-024752_0_2_0_2_75_000_200.json (0.0MB)
      K-018357-020238-024752_0_2_0_2_90_000_200.json (0.0MB)
    K-020238/
      K-018357-020238-024752_0_2_0_2_70_000_200.json (0.0MB)
      K-018357-020238-024752_0_2_0_2_75_000_200.json (0.0MB)
      K-018357-020238-024752_0_2_0_2_90_000_200.json (0.0MB)
  K-018357-020238-028424_json/
    K-018357/
      K-018357-020238-028424_0_2_0_2_70_000_200.json (0.0MB)
      K-018357-020238-028424_0_2_0_2_75_000_200.json (0.0MB)
      K-018357-020238-028424_0_2_0_2_90_000_200.json (0.0MB)
    K-020238/
      K-018357-020238-028424_0_2_0_2_70_000_200.json (0.0MB)
      K-018357-020238-028424_0_2_0_2_75_000_200.json (0.0MB)
      K-018357-020238-028424_0_2_0_2_90_000_200.json (0.0MB)
    K-028424/
      K-018357-020238-028424_0_2_0_2_70_000_200.json (0.0MB)
      K-0

In [9]:
import glob

# 전체 파일 탐색
all_files = glob.glob(os.path.join(AIHUB_DIR, '**', '*'), recursive=True)
imgs  = [f for f in all_files if f.endswith(('.png','.jpg'))]
jsons = [f for f in all_files if f.endswith('.json')]

print(f'이미지: {len(imgs)}장')
print(f'JSON  : {len(jsons)}개')

print('\n이미지 샘플:')
for f in imgs[:3]: print(f'  {f}')

print('\nJSON 샘플:')
for f in jsons[:3]: print(f'  {f}')

이미지: 2000장
JSON  : 5520개

이미지 샘플:
  /content/aihub_pills/K-018147-020238-033026/K-018147-020238-033026_index.png
  /content/aihub_pills/K-018147-020238-033026/K-018147-020238-033026_0_2_0_2_75_000_200.png
  /content/aihub_pills/K-018147-020238-033026/K-018147-020238-033026_0_2_0_2_70_000_200.png

JSON 샘플:
  /content/aihub_pills/K-016262-019232-020259_json/K-016262/K-016262-019232-020259_0_2_0_2_90_000_200.json
  /content/aihub_pills/K-016262-019232-020259_json/K-016262/K-016262-019232-020259_0_2_0_2_70_000_200.json
  /content/aihub_pills/K-016262-019232-020259_json/K-016262/K-016262-019232-020259_0_2_0_2_75_000_200.json


In [10]:
import subprocess

# tar 내용물 확인
result = subprocess.run(['tar', '-tvf', '/content/VL_image.tar'],
                        capture_output=True, text=True)
print('VL_image.tar 내용 (앞 20줄):')
for line in result.stdout.split('\n')[:20]:
    print(line)

print('\nVL_image_json.tar 내용 (앞 20줄):')
result2 = subprocess.run(['tar', '-tvf', '/content/VL_image_json.tar'],
                         capture_output=True, text=True)
for line in result2.stdout.split('\n')[:20]:
    print(line)

VL_image.tar 내용 (앞 20줄):
-rw-r--r-- 0/0      1073741824 2026-07-03 02:44 166.약품식별_인공지능_개발을_위한_경구약제_이미지_데이터/01.데이터/2.Validation/원천데이터/경구약제조합_5000종/VS_1_조합.zip.part0
-rw-r--r-- 0/0      1073741824 2026-07-03 02:50 166.약품식별_인공지능_개발을_위한_경구약제_이미지_데이터/01.데이터/2.Validation/원천데이터/경구약제조합_5000종/VS_1_조합.zip.part1073741824
-rw-r--r-- 0/0      1073741824 2026-07-03 02:56 166.약품식별_인공지능_개발을_위한_경구약제_이미지_데이터/01.데이터/2.Validation/원천데이터/경구약제조합_5000종/VS_1_조합.zip.part2147483648
-rw-r--r-- 0/0       376564524 2026-07-03 03:02 166.약품식별_인공지능_개발을_위한_경구약제_이미지_데이터/01.데이터/2.Validation/원천데이터/경구약제조합_5000종/VS_1_조합.zip.part3221225472


VL_image_json.tar 내용 (앞 20줄):
-rw-r--r-- 0/0         7323186 2026-07-03 02:43 166.약품식별_인공지능_개발을_위한_경구약제_이미지_데이터/01.데이터/2.Validation/라벨링데이터/경구약제조합_5000종/VL_1_조합.zip.part0



In [11]:
import glob, os, subprocess

BASE = os.path.join(AIHUB_DIR,
    '166.약품식별_인공지능_개발을_위한_경구약제_이미지_데이터/01.데이터')

# ── 이미지: VS_1_조합.zip 합치기 ─────────────────────────────
IMG_PARTS_DIR = os.path.join(BASE, '2.Validation/원천데이터/경구약제조합_5000종')
img_parts = sorted(glob.glob(os.path.join(IMG_PARTS_DIR, 'VS_1_조합.zip.part*')))
print(f'이미지 파트: {len(img_parts)}개')

VS_ZIP = os.path.join(IMG_PARTS_DIR, 'VS_1_조합.zip')
if not os.path.exists(VS_ZIP):
    print('파트 합치는 중...')
    with open(VS_ZIP, 'wb') as out:
        for part in img_parts:
            print(f'  {os.path.basename(part)}')
            with open(part, 'rb') as f:
                out.write(f.read())
    print('합치기 완료')

print('이미지 zip 압축 해제 중...')
!unzip -q {VS_ZIP} -d {AIHUB_DIR}
print('✅ 이미지 완료')

# ── JSON: VL_1_조합.zip 합치기 ───────────────────────────────
JSON_PARTS_DIR = os.path.join(BASE, '2.Validation/라벨링데이터/경구약제조합_5000종')
json_parts = sorted(glob.glob(os.path.join(JSON_PARTS_DIR, 'VL_1_조합.zip.part*')))
print(f'\nJSON 파트: {len(json_parts)}개')

VL_ZIP = os.path.join(JSON_PARTS_DIR, 'VL_1_조합.zip')
if not os.path.exists(VL_ZIP):
    with open(VL_ZIP, 'wb') as out:
        for part in json_parts:
            with open(part, 'rb') as f:
                out.write(f.read())

print('JSON zip 압축 해제 중...')
!unzip -q {VL_ZIP} -d {AIHUB_DIR}
print('✅ JSON 완료')

# ── 결과 확인 ─────────────────────────────────────────────────
imgs  = glob.glob(os.path.join(AIHUB_DIR,'**','*.png'), recursive=True)
jsons = glob.glob(os.path.join(AIHUB_DIR,'**','*.json'), recursive=True)
jsons = [j for j in jsons if not any(fb in j for fb in FORBIDDEN)]
print(f'\n이미지: {len(imgs)}장')
print(f'JSON  : {len(jsons)}개')

이미지 파트: 4개
이미지 zip 압축 해제 중...
replace /content/aihub_pills/K-016235-027733-029667-031885/K-016235-027733-029667-031885_0_2_0_2_70_000_200.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: 
error:  invalid response [{ENTER}]
replace /content/aihub_pills/K-016235-027733-029667-031885/K-016235-027733-029667-031885_0_2_0_2_70_000_200.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: 
error:  invalid response [{ENTER}]
replace /content/aihub_pills/K-016235-027733-029667-031885/K-016235-027733-029667-031885_0_2_0_2_70_000_200.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: 
error:  invalid response [{ENTER}]
replace /content/aihub_pills/K-016235-027733-029667-031885/K-016235-027733-029667-031885_0_2_0_2_70_000_200.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: 
error:  invalid response [{ENTER}]
replace /content/aihub_pills/K-016235-027733-029667-031885/K-016235-027733-029667-031885_0_2_0_2_70_000_200.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: ✅ 이미지 완료

JSON 파트: 1개
JSON zip 압축 해제 중...
replace /content/aihub_pills/

## 4. 라벨 파싱 → 약품명 확인

다운로드된 라벨 JSON에서 약품명을 추출합니다.


In [12]:
import subprocess
import matplotlib.font_manager as fm
subprocess.run(['apt-get','install','-y','-q','fonts-nanum'], capture_output=True)
fm.fontManager.__init__()
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
print('✅ 한글 폰트 설정 완료')

✅ 한글 폰트 설정 완료


In [13]:
print('FORBIDDEN:', FORBIDDEN)

# FORBIDDEN 없이 전체 JSON 탐색
all_jsons = glob.glob(os.path.join(AIHUB_DIR,'**','*.json'), recursive=True)
print(f'전체 JSON: {len(all_jsons)}개')

# FORBIDDEN 필터 적용
filtered = [j for j in all_jsons if not any(fb in j for fb in FORBIDDEN)]
print(f'필터 후  : {len(filtered)}개')

# label_jsons 재설정
label_jsons = filtered
print(f'\nlabel_jsons 재설정 완료: {len(label_jsons)}개')

FORBIDDEN: ['TL_2_조합', 'TS_2_조합']
전체 JSON: 5520개
필터 후  : 5520개

label_jsons 재설정 완료: 5520개


In [14]:
from collections import defaultdict
import glob, json, os

drug_info = defaultdict(lambda: {'name':'', 'idx':'', 'samples':[]})

for jp in label_jsons:
    try:
        with open(jp, encoding='utf-8') as f:
            d = json.load(f)
        img_meta = d['images'][0]
        drug_n   = img_meta.get('drug_N', '')
        dl_name  = img_meta.get('dl_name', '?')
        dl_idx   = img_meta.get('dl_idx', '')

        info = drug_info[drug_n]
        info['name'] = dl_name
        info['idx']  = dl_idx
    except: pass

print(f'✅ drug_info 생성 완료: {len(drug_info)}개 클래스')
for drug_n, info in list(drug_info.items())[:3]:
    print(f'  {drug_n} → {info["name"]} (idx={info["idx"]})')

✅ drug_info 생성 완료: 54개 클래스
  K-016262 → 크레스토정 20mg (idx=16261)
  K-019232 → 콜리네이트연질캡슐 400mg (idx=19231)
  K-020259 → 자트랄엑스엘정 10mg (idx=20258)


In [15]:
# AIHUB_DIR 확인
import os, glob
print('AIHUB_DIR:', AIHUB_DIR)
all_files = glob.glob(os.path.join(AIHUB_DIR,'**','*'), recursive=True)
print(f'전체 파일: {len(all_files)}개')
jsons = [f for f in all_files if f.endswith('.json')]
print(f'JSON 파일: {len(jsons)}개')
for j in jsons[:3]: print(f'  {j}')

AIHUB_DIR: /content/aihub_pills
전체 파일: 10374개
JSON 파일: 5520개
  /content/aihub_pills/K-016262-019232-020259_json/K-016262/K-016262-019232-020259_0_2_0_2_90_000_200.json
  /content/aihub_pills/K-016262-019232-020259_json/K-016262/K-016262-019232-020259_0_2_0_2_70_000_200.json
  /content/aihub_pills/K-016262-019232-020259_json/K-016262/K-016262-019232-020259_0_2_0_2_75_000_200.json


In [16]:
print('FORBIDDEN:', FORBIDDEN)

# FORBIDDEN 없이 전체 JSON 탐색
all_jsons = glob.glob(os.path.join(AIHUB_DIR,'**','*.json'), recursive=True)
print(f'전체 JSON: {len(all_jsons)}개')

# FORBIDDEN 필터 적용
filtered = [j for j in all_jsons if not any(fb in j for fb in FORBIDDEN)]
print(f'필터 후  : {len(filtered)}개')

# label_jsons 재설정
label_jsons = filtered
print(f'\nlabel_jsons 재설정 완료: {len(label_jsons)}개')

FORBIDDEN: ['TL_2_조합', 'TS_2_조합']
전체 JSON: 5520개
필터 후  : 5520개

label_jsons 재설정 완료: 5520개


In [17]:
def drug_n_to_cat_id(drug_n):
    # 'K-016262' → 16262
    try:
        return int(drug_n.replace('K-0', '').replace('K-', ''))
    except:
        return None

# 재비교
aihub_cat_ids = set()
for drug_n in drug_info.keys():
    cid = drug_n_to_cat_id(drug_n)
    if cid:
        aihub_cat_ids.add(cid)

orig_cat_ids = {1900,2483,3351,3483,3544,3743,3832,4543,
                12081,12247,12778,13395,13900,16232,16262,
                16548,16551,16688,18147,18357,19232,19552,
                19607,19861,20014,20238,20877,21325,21771,
                22074,22347,22362,24850,25367,25438,25469,
                27733,27777,27926,27993,28763,29345,29451,
                29667,30308,31863,31885,32310,33009,33208,
                33880,34597,35206,36637,38162,41768}

overlap  = aihub_cat_ids & orig_cat_ids
new_only = aihub_cat_ids - orig_cat_ids

print(f'겹치는 클래스  : {len(overlap)}개 → 기존 데이터 보강')
print(f'AI Hub에만 있는: {len(new_only)}개 → 신규 추가 가능')
print(f'\n겹치는 cat_id: {sorted(overlap)}')
print(f'신규 cat_id: {sorted(new_only)[:10]}...')

겹치는 클래스  : 39개 → 기존 데이터 보강
AI Hub에만 있는: 15개 → 신규 추가 가능

겹치는 cat_id: [16262, 16548, 16551, 16688, 18147, 18357, 19232, 19607, 19861, 20014, 20238, 20877, 21325, 21771, 22074, 22347, 24850, 25367, 25469, 27733, 27777, 27926, 27993, 28763, 29345, 29451, 29667, 30308, 31863, 31885, 32310, 33009, 33208, 33880, 34597, 35206, 36637, 38162, 41768]
신규 cat_id: [16235, 18110, 20259, 20852, 21026, 23203, 24752, 27653, 28424, 29871]...


# 💊 AI Hub 데이터 병합 + 재학습

```
기존 56클래스 232장
+ AI Hub 39개 겹치는 클래스 (데이터 보강)
+ AI Hub 15개 신규 클래스
= 71클래스 재학습

그룹 기반 5폴드: 같은 약 조합(각도만 다른 이미지) → 같은 폴드
```


## 1. AI Hub COCO 변환 + 클래스 매핑

In [18]:
# ── AI Hub dl_idx → cat_id 매핑 ─────────────────────────────
def drug_n_to_cat_id(drug_n):
    try: return int(drug_n.replace('K-0','').replace('K-',''))
    except: return None

def get_group_key(fname):
    base = os.path.basename(fname)
    if '_0_2' in base: return base.split('_0_2')[0]
    return base.split('.')[0]

# 기존 클래스 매핑 (1~56)
orig_cat_ids = sorted({1900,2483,3351,3483,3544,3743,3832,4543,
    12081,12247,12778,13395,13900,16232,16262,16548,16551,16688,
    18147,18357,19232,19552,19607,19861,20014,20238,20877,21325,
    21771,22074,22347,22362,24850,25367,25438,25469,27733,27777,
    27926,27993,28763,29345,29451,29667,30308,31863,31885,32310,
    33009,33208,33880,34597,35206,36637,38162,41768})
orig_cat2label = {c: i+1 for i,c in enumerate(orig_cat_ids)}

# AI Hub 신규 클래스 (57~)
aihub_new_map = {}
next_label = len(orig_cat2label) + 1
for drug_n in sorted(drug_info.keys()):
    cid = drug_n_to_cat_id(drug_n)
    if cid and cid not in orig_cat2label and cid not in aihub_new_map:
        aihub_new_map[cid] = next_label; next_label += 1

TOTAL_CLASSES = next_label - 1
final_label2cat = {v:k for k,v in orig_cat2label.items()}
final_label2cat.update({v:k for k,v in aihub_new_map.items()})
print(f'기존 56 + 신규 {len(aihub_new_map)} = 총 {TOTAL_CLASSES}개 클래스')

# ── AI Hub → COCO 변환 ───────────────────────────────────────
AIHUB_IMG_OUT = '/content/aihub_processed/images'
AIHUB_ANN_OUT = '/content/aihub_processed/annotations.json'
os.makedirs(AIHUB_IMG_OUT, exist_ok=True)

img_path_dict = {}
for root,_,files in os.walk(AIHUB_DIR):
    for f in files:
        if f.lower().endswith(('.png','.jpg','.jpeg')):
            img_path_dict[f] = os.path.join(root, f)

cats_coco = [{'id':0,'name':'pill','supercategory':'none'}]
cats_coco += [{'id':lbl,'name':str(cat),'supercategory':'pill'}
              for lbl,cat in sorted(final_label2cat.items())]

aihub_images, aihub_anns = [], []
img_id=1; ann_id=1; skipped=0

for jp in label_jsons:
    try:
        with open(jp, encoding='utf-8') as f: d = json.load(f)
        img_meta = d['images'][0]
        fname    = os.path.basename(img_meta['file_name'])
        drug_n   = img_meta.get('drug_N','')
        dl_name  = img_meta.get('dl_name','?')
        cid      = drug_n_to_cat_id(drug_n)
        if cid is None: continue
        label = orig_cat2label.get(cid) or aihub_new_map.get(cid)
        if label is None: continue
        ann = d['annotations'][0] if d.get('annotations') else None
        if ann is None: continue
        src = img_path_dict.get(fname)
        if src is None: skipped+=1; continue
        dst = os.path.join(AIHUB_IMG_OUT, fname)
        if not os.path.exists(dst): shutil.copy(src, dst)
        aihub_images.append({'id':img_id,'file_name':fname,
            'width':img_meta.get('width',1200),'height':img_meta.get('height',1200),
            'dl_name':dl_name,'drug_N':drug_n})
        x,y,w,h = ann['bbox']
        aihub_anns.append({'id':ann_id,'image_id':img_id,'category_id':label,
            'bbox':[float(x),float(y),float(w),float(h)],'area':float(w*h),'iscrowd':0})
        img_id+=1; ann_id+=1
    except: pass

aihub_coco = {'images':aihub_images,'annotations':aihub_anns,'categories':cats_coco}
with open(AIHUB_ANN_OUT,'w',encoding='utf-8') as f:
    json.dump(aihub_coco, f, ensure_ascii=False, indent=2)
print(f'AI Hub COCO 변환: {len(aihub_images)}장 / {len(aihub_anns)}박스 (스킵 {skipped}장)')

기존 56 + 신규 15 = 총 71개 클래스
AI Hub COCO 변환: 5520장 / 5512박스 (스킵 0장)


## 2. 기존 Train 로드 + DataEngineer 패치

In [19]:
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np

TRAIN_ANN = f'{SHARED_DIR}/sprint_ai_project1_data/train_annotations'
TRAIN_IMG  = f'{SHARED_DIR}/sprint_ai_project1_data/train_images'

COORD_FIX = {'K-003351-016262-018357_0_2_0_2_75_000_200.png':[([6567,625,311,315],[567,625,311,315])]}
ADD_BOXES = {
    'K-001900-016548-019607-033009_0_2_0_2_70_000_200.png':[(16548,[90,870,245,240])],
    'K-003351-013900-021325_0_2_0_2_70_000_200.png':[(3351,[400,830,180,180])],
    'K-003351-013900-036637_0_2_0_2_70_000_200.png':[(3351,[440,880,175,175])],
    'K-003351-020014-022074_0_2_0_2_90_000_200.png':[(20014,[65,720,325,315])],
    'K-003351-020238-031863_0_2_0_2_70_000_200.png':[(3351,[580,290,215,215])],
    'K-003351-021325-032310_0_2_0_2_90_000_200.png':[(32310,[595,830,345,245])],
    'K-003351-029667-031863_0_2_0_2_70_000_200.png':[(3351,[375,870,165,165])],
    'K-003351-032310-038162_0_2_0_2_70_000_200.png':[(3351,[390,855,185,185])],
    'K-003351-033880-038162_0_2_0_2_75_000_200.png':[(33880,[70,600,310,425])],
    'K-003351-035206-041768_0_2_0_2_70_000_200.png':[(3351,[460,875,180,180])],
    'K-003544-004543-012247-016548_0_2_0_2_90_000_200.png':[(4543,[640,195,205,190])],
}
REMOVE_BOXES = {'K-001900-016548-019607-033009_0_2_0_2_70_000_200.png':[(16548,[88,255,366,209])]}
MODIFY_BOXES = {
    'K-003351-020014-020238_0_2_0_2_90_000_200.png':[(3351,None,[390,260,170,165])],
    'K-003351-019232-029667_0_2_1_2_70_000_200.png':[(19232,None,'EXTEND_DOWN_95')],
}

boxes_by_fname = defaultdict(list)
cats_by_fname  = defaultdict(list)
img_meta_orig  = {}
for jp in glob.glob(os.path.join(TRAIN_ANN,'**','*.json'),recursive=True):
    with open(jp,encoding='utf-8') as f: d=json.load(f)
    im=d['images'][0]; fn=im['file_name']
    img_meta_orig[fn]=(im['width'],im['height'])
    for ann in d['annotations']:
        boxes_by_fname[fn].append(ann['bbox']); cats_by_fname[fn].append(ann['category_id'])

for fn,fixes in COORD_FIX.items():
    for w,r in fixes:
        for i,b in enumerate(boxes_by_fname[fn]):
            if b==w: boxes_by_fname[fn][i]=r
for fn,removes in REMOVE_BOXES.items():
    for rc,rb in removes:
        kb,kc,done=[],[],False
        for c,b in zip(cats_by_fname[fn],boxes_by_fname[fn]):
            if (not done) and c==rc and b==rb: done=True; continue
            kb.append(b); kc.append(c)
        boxes_by_fname[fn],cats_by_fname[fn]=kb,kc
for fn,mods in MODIFY_BOXES.items():
    for mc,w,new in mods:
        for i,(c,b) in enumerate(zip(cats_by_fname[fn],boxes_by_fname[fn])):
            if c==mc and (w is None or b==w):
                boxes_by_fname[fn][i]=([b[0],b[1],b[2],b[3]+95] if new=='EXTEND_DOWN_95' else new); break
for fn,adds in ADD_BOXES.items():
    for ac,ab in adds: cats_by_fname[fn].append(ac); boxes_by_fname[fn].append(ab)

print(f'✅ 기존: {len(boxes_by_fname)}장 / {sum(len(v) for v in boxes_by_fname.values())}박스 (패치 완료)')

✅ 기존: 232장 / 773박스 (패치 완료)


## 3. 전체 병합 + 그룹 기반 5폴드 생성

같은 약 조합(각도만 다른 이미지)은 항상 같은 폴드로 배치됩니다.

In [20]:
# ── 전체 병합 ────────────────────────────────────────────────
all_img_paths = {}
for root,_,files in os.walk(TRAIN_IMG):
    for f in files:
        if f.lower().endswith(('.png','.jpg','.jpeg')): all_img_paths[f]=os.path.join(root,f)
for root,_,files in os.walk(AIHUB_IMG_OUT):
    for f in files:
        if f.lower().endswith(('.png','.jpg','.jpeg')): all_img_paths[f]=os.path.join(root,f)

all_boxes,all_cats,all_meta = {},{},{}
for fn in boxes_by_fname:
    lbls = [orig_cat2label[c] for c in cats_by_fname[fn] if c in orig_cat2label]
    if not lbls: continue
    all_boxes[fn]=[list(b) for b in boxes_by_fname[fn]]; all_cats[fn]=lbls; all_meta[fn]=img_meta_orig[fn]

aihub_ann_by = defaultdict(list)
for ann in aihub_anns: aihub_ann_by[ann['image_id']].append(ann)
for img_info in aihub_images:
    fname=img_info['file_name']; anns=aihub_ann_by[img_info['id']]
    if not anns: continue
    all_boxes[fname]=[a['bbox'] for a in anns]
    all_cats[fname]=[a['category_id'] for a in anns]
    all_meta[fname]=(img_info['width'],img_info['height'])

all_fnames=sorted(all_boxes.keys())
print(f'전체: {len(all_fnames)}장 / {sum(len(v) for v in all_boxes.values())}박스')

# ── 그룹 기반 5폴드 (같은 약 조합 = 같은 폴드) ─────────────
SEED=42; N_SPLITS=5
random.seed(SEED); np.random.seed(SEED)

cls_freq = Counter(c for cs in all_cats.values() for c in cs)
groups   = np.array([get_group_key(fn) for fn in all_fnames])
strat    = np.array([min(all_cats[fn],key=lambda c:cls_freq[c]) for fn in all_fnames])
folds    = list(StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=SEED).split(all_fnames,strat,groups))

for fi,(tr,va) in enumerate(folds):
    g_train=set(groups[tr]); g_val=set(groups[va])
    print(f'fold{fi}: train={len(tr)} / valid={len(va)} | 그룹누수={len(g_train&g_val)}개')

# fold 디렉토리 생성
DATASET_DIR_V2='/content/dataset_v2'
for fi,(tr,va) in enumerate(folds):
    for idxs,split in [(tr,'train'),(va,'valid')]:
        files=[all_fnames[i] for i in idxs]
        d=os.path.join(DATASET_DIR_V2,f'fold{fi}',split); os.makedirs(d,exist_ok=True)
        imgs_c,anns_c,aid=[],[],1
        for iid,fn in enumerate(files,1):
            W,H=all_meta.get(fn,(1200,1200))
            imgs_c.append({'id':iid,'file_name':os.path.basename(fn),'width':W,'height':H})
            for lbl,bbox in zip(all_cats[fn],all_boxes[fn]):
                x,y,w,h=bbox
                anns_c.append({'id':aid,'image_id':iid,'category_id':lbl,
                    'bbox':[float(x),float(y),float(w),float(h)],'area':float(w*h),'iscrowd':0}); aid+=1
            p=all_img_paths.get(os.path.basename(fn))
            if p and os.path.exists(p): shutil.copy(p,os.path.join(d,os.path.basename(fn)))
        json.dump({'images':imgs_c,'annotations':anns_c,'categories':cats_coco},
                  open(os.path.join(d,'_annotations.coco.json'),'w'))
    print(f'fold{fi} 완료')

json.dump({'label2cat':{str(k):v for k,v in final_label2cat.items()},'num_classes':TOTAL_CLASSES},
          open(os.path.join(DATASET_DIR_V2,'label_map.json'),'w'))
print(f'✅ 5폴드 생성 완료 ({TOTAL_CLASSES}클래스)')

전체: 1732장 / 2273박스
fold0: train=1379 / valid=353 | 그룹누수=0개
fold1: train=1386 / valid=346 | 그룹누수=0개
fold2: train=1393 / valid=339 | 그룹누수=0개


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:1023: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


fold3: train=1381 / valid=351 | 그룹누수=0개
fold4: train=1389 / valid=343 | 그룹누수=0개
fold0 완료
fold1 완료
fold2 완료
fold3 완료
fold4 완료
✅ 5폴드 생성 완료 (71클래스)


## 4. Copy-Paste 균형 증강

In [21]:
import sys, importlib
sys.path.insert(0,'/content')
import augment_dataset; importlib.reload(augment_dataset)
from augment_dataset import augment_fold

AUG_DIR_V2='/content/dataset_v2_aug'
AUG_FACTOR=2

for fi in range(N_SPLITS):
    dst=os.path.join(AUG_DIR_V2,f'fold{fi}')
    if os.path.exists(dst) and os.listdir(dst): print(f'fold{fi}: 스킵'); continue
    print(f'fold{fi} 균형 증강 중...')
    augment_fold('copy_paste_balanced',fold_idx=fi,
                 dataset_dir=DATASET_DIR_V2,out_dir=AUG_DIR_V2,aug_factor=AUG_FACTOR)

print('\n✅ 증강 완료')
for fi in range(N_SPLITS):
    d=os.path.join(AUG_DIR_V2,f'fold{fi}','train')
    if os.path.exists(d):
        n=len([f for f in os.listdir(d) if f.endswith('.png')])
        print(f'  fold{fi}: {n}장')

fold0: 스킵
fold1: 스킵
fold2: 스킵
fold3: 스킵
fold4: 스킵

✅ 증강 완료
  fold0: 4137장
  fold1: 4158장
  fold2: 4179장
  fold3: 4143장
  fold4: 4167장


/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()
/content/augment_dataset.py:130: UserWarning: Argument(s) 'shift_limit' are not valid for transform OpticalDistortion
  A.OpticalDistortion(distort_limit=0.2,
/content/augment_dataset.py:136: UserWarning: Argument(s) 'slant_lower, slant_upper' are not valid for transform RandomRain
  A.RandomRain(slant_lower=-10, slant_upper=10,
/content/augment_dataset.py:140: UserWarning: Argument(s) 'fog_coef_lower, fog_coef_upper' are not valid for transform RandomFog
  A.RandomFog(fog_coef_lower=0.1, fog_coef_upper=0.35,
/content/augment_dataset.py:142: UserWarning: Argument(s) 'num_shadows_lower, num_shadows_upper' are not valid for transform RandomShadow
  A.RandomShadow(shadow_roi=(0, 0, 1, 1),
/content/augment_dataset.py:145: UserWarning: Argument(s) 'num_flare_circles_lower, num_flare_circles_upper' are not valid for transfor

In [22]:
## 5. RF-DETR Medium 5폴드 학습

In [23]:
import torch, os, glob, shutil, time, threading, csv, json
from rfdetr import RFDETRMedium

MY_OUTPUT  = '/content/drive/MyDrive/pill_detection_outputs'
EXP_NAME   = 'final_v2_aihub'
METRICS_DIR = os.path.join(MY_OUTPUT, 'metrics')
os.makedirs(METRICS_DIR, exist_ok=True)

LR,LR_ENCODER,WEIGHT_DECAY        = 1e-4,1.5e-4,1e-4
LR_SCHEDULER,WARMUP_EPOCHS,LR_MIN = 'cosine',0.0,0.0
EPOCHS,ES_PATIENCE,ES_MIN_DELTA   = 100,10,0.001
BATCH_SIZE = 8 if torch.cuda.get_device_properties(0).total_memory/1e9>=16 else 4
GRAD_ACCUM = 2 if BATCH_SIZE==8 else 4

print(f'Early Stopping : patience={ES_PATIENCE}, min_delta={ES_MIN_DELTA}')
print(f'Train/Val 분리 : {AUG_DIR_V2}/fold{{fi}}/train | valid')
print(f'메트릭 저장    : {METRICS_DIR}')
print(f'배치           : {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM}')

# ── Drive 메트릭 백그라운드 저장 클래스 ──────────────────────
class DriveMetricsSaver:
    """
    백그라운드 스레드로 TensorBoard 로그를 읽어
    Google Drive에 CSV로 저장합니다.
    세션이 끊겨도 Drive에 남아있습니다.
    """
    def __init__(self, output_dir, csv_path, fold_idx, interval=60):
        self.output_dir = output_dir
        self.csv_path   = csv_path
        self.fold_idx   = fold_idx
        self.interval   = interval
        self._stop      = threading.Event()
        self._saved_eps = set()

    def start(self):
        self._stop.clear()
        self._t = threading.Thread(target=self._loop, daemon=True)
        self._t.start()
        print(f'  📊 메트릭 저장 시작 → {self.csv_path}')

    def stop(self):
        self._stop.set()
        self._save()  # 종료 전 마지막 저장
        print(f'  📊 메트릭 저장 완료')

    def _loop(self):
        while not self._stop.is_set():
            self._save()
            self._stop.wait(self.interval)

    def _save(self):
        try:
            from tensorboard.backend.event_processing.event_accumulator \
                import EventAccumulator
            logs = sorted(glob.glob(
                os.path.join(self.output_dir,'**','events.out.*'),
                recursive=True))
            if not logs: return

            ea = EventAccumulator(logs[-1]); ea.Reload()
            tags = ea.Tags().get('scalars', [])

            # 태그 자동 탐지
            tag_map = {}
            for t in tags:
                tl = t.lower()
                if 'train' in tl and 'loss' in tl and 'train_loss' not in tag_map:
                    tag_map['train_loss'] = t
                elif 'lr' in tl and 'lr' not in tag_map:
                    tag_map['lr'] = t
                elif ('map_50' in tl or 'map50' in tl) and 'mAP50' not in tag_map:
                    tag_map['mAP50'] = t
                elif ('map_75' in tl or 'map75' in tl) and 'mAP75' not in tag_map:
                    tag_map['mAP75'] = t
                elif 'map' in tl and '0.5' in tl and 'mAP' not in tag_map:
                    tag_map['mAP'] = t

            # epoch별 데이터 수집
            rows = {}
            for metric, tag in tag_map.items():
                for s in ea.Scalars(tag):
                    ep = s.step
                    if ep not in rows:
                        rows[ep] = {'fold':self.fold_idx,'epoch':ep}
                    rows[ep][metric] = round(s.value, 6)

            # 새로운 epoch만 추가
            new_rows = [rows[ep] for ep in sorted(rows)
                        if ep not in self._saved_eps]
            if not new_rows: return

            fields = ['fold','epoch','train_loss','lr','mAP50','mAP75','mAP']
            write_header = not os.path.exists(self.csv_path)
            with open(self.csv_path, 'a', newline='') as f:
                w = csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
                if write_header: w.writeheader()
                for row in new_rows:
                    w.writerow(row)
                    self._saved_eps.add(row['epoch'])

            print(f'  💾 epoch {max(self._saved_eps)} 저장됨 '
                  f'(총 {len(self._saved_eps)}개)')
        except Exception as e:
            pass  # 백그라운드 오류는 무시

# ── 5폴드 학습 루프 ──────────────────────────────────────────
train_results = []

for fi in range(N_SPLITS):
    exp      = f'{EXP_NAME}_fold{fi}'
    dst_pth  = os.path.join(MY_OUTPUT,  f'{exp}_best.pth')
    csv_path = os.path.join(METRICS_DIR, f'{exp}_metrics.csv')
    out_dir  = f'/content/outputs/{exp}'

    if os.path.exists(dst_pth):
        print(f'[fold {fi}] ✅ 완료됨 → 스킵')
        train_results.append({'fold':fi,'status':'skipped'}); continue

    os.makedirs(out_dir, exist_ok=True)
    print(f"\n{'='*55}")
    print(f'[fold {fi}] 학습 시작 | 클래스={TOTAL_CLASSES}')
    print(f"{'='*55}")

    # 메트릭 저장 시작
    saver = DriveMetricsSaver(out_dir, csv_path, fold_idx=fi, interval=60)
    saver.start()

    try:
        model = RFDETRMedium()
        model.train(
            dataset_dir              = f'{AUG_DIR_V2}/fold{fi}',
            output_dir               = out_dir,
            epochs                   = EPOCHS,
            batch_size               = BATCH_SIZE,
            grad_accum_steps         = GRAD_ACCUM,
            lr                       = LR,
            lr_encoder               = LR_ENCODER,
            weight_decay             = WEIGHT_DECAY,
            lr_scheduler             = LR_SCHEDULER,
            warmup_epochs            = WARMUP_EPOCHS,
            lr_min_factor            = LR_MIN,
            early_stopping           = True,
            early_stopping_patience  = ES_PATIENCE,
            early_stopping_min_delta = ES_MIN_DELTA,
            tensorboard              = True,
        )
        # best 체크포인트 → Drive 저장
        src = os.path.join(out_dir, 'checkpoint_best_total.pth')
        if os.path.exists(src):
            shutil.copy(src, dst_pth)
            print(f'[fold {fi}] ✅ best 저장 → {dst_pth}')
            train_results.append({'fold':fi,'status':'done','ckpt':dst_pth})
        else:
            train_results.append({'fold':fi,'status':'no_checkpoint'})

    except Exception as e:
        print(f'[fold {fi}] ❌ {e}')
        train_results.append({'fold':fi,'status':f'error:{str(e)[:60]}'})

    finally:
        saver.stop()
        try: del model
        except: pass
        torch.cuda.empty_cache()
        print(f'[fold {fi}] GPU 정리 완료')

# ── 결과 요약 ─────────────────────────────────────────────────
import pandas as pd
print(f"\n{'='*55}\n✅ 5폴드 학습 완료\n{'='*55}")
display(pd.DataFrame(train_results))

# 전체 메트릭 CSV 합치기
all_metrics = []
for fi in range(N_SPLITS):
    csv_path = os.path.join(METRICS_DIR, f'{EXP_NAME}_fold{fi}_metrics.csv')
    if os.path.exists(csv_path):
        with open(csv_path) as f:
            reader = csv.DictReader(f)
            all_metrics.extend(list(reader))

if all_metrics:
    merged_path = os.path.join(METRICS_DIR, f'{EXP_NAME}_all_metrics.csv')
    with open(merged_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=all_metrics[0].keys())
        writer.writeheader(); writer.writerows(all_metrics)
    print(f'📊 전체 메트릭 저장 → {merged_path}')
    display(pd.read_csv(merged_path))


/usr/local/lib/python3.12/dist-packages/rfdetr/models/weights.py:258: FutureWarning: target=True is deprecated since `v0.8`; use `TargetMode.ARGS_REMAP` instead. Will be removed in `v1.0`.
  @deprecated(target=True, args_mapping={"train_config": None}, deprecated_in="1.7.0", remove_in="1.9.0", num_warns=-1)


Early Stopping : patience=10, min_delta=0.001
Train/Val 분리 : /content/dataset_v2_aug/fold{fi}/train | valid
메트릭 저장    : /content/drive/MyDrive/pill_detection_outputs/metrics
배치           : 8 x 2 = 16
[fold 0] ✅ 완료됨 → 스킵
[fold 1] ✅ 완료됨 → 스킵

[fold 2] 학습 시작 | 클래스=71
  📊 메트릭 저장 시작 → /content/drive/MyDrive/pill_detection_outputs/metrics/final_v2_aihub_fold2_metrics.csv
  💾 epoch 4799 저장됨 (총 114개)
[2026-07-03 12:29:10] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-medium.pth already exists with correct MD5 hash.


[2026-07-03 12:29:10] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-07-03 12:29:10] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-07-03 12:29:12] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-medium.pth already exists with correct MD5 hash.


[2026-07-03 12:29:13] [WARNING] rf-detr - Pretrained weights at '/root/.roboflow/models/rf-detr-medium.pth' loaded only partially — this typically produces lower accuracy. 1 model parameter(s) not in checkpoint (left at random init): [_kp_active_mask]. Check that the model configuration (encoder, hidden_dim, out_feature_indexes, projector_scale, ...) matches the architecture the checkpoint was trained with.
[2026-07-03 12:29:13] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-07-03 12:29:13] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


...

INFO:pytorch_lightning.callbacks.early_stopping:Monitored metric __rfdetr_effective_map__ did not improve in the last 10 records. Best score: 0.993. Signaling Trainer to stop.


[2026-07-03 17:03:33] [INFO] rf-detr - Best total checkpoint saved from EMA (regular=0.9884, ema=0.9936)
[fold 4] ✅ best 저장 → /content/drive/MyDrive/pill_detection_outputs/final_v2_aihub_fold4_best.pth
  💾 epoch 6524 저장됨 (총 155개)
  📊 메트릭 저장 완료
[fold 4] GPU 정리 완료

✅ 5폴드 학습 완료


,fold,status,ckpt
0,0,skipped,NaN
1,1,skipped,NaN
2,2,done,/content/drive/MyDrive/pill_detection_outputs/...
3,3,done,/content/drive/MyDrive/pill_detection_outputs/...
4,4,done,/content/drive/MyDrive/pill_detection_outputs/...


📊 전체 메트릭 저장 → /content/drive/MyDrive/pill_detection_outputs/metrics/final_v2_aihub_all_metrics.csv


,fold,epoch,train_loss,lr,mAP50,mAP75,mAP
0,0,49,NaN,0.000100,NaN,NaN,NaN
1,0,99,NaN,0.000100,NaN,NaN,NaN
2,0,149,NaN,0.000100,NaN,NaN,NaN
3,0,199,NaN,0.000100,NaN,NaN,NaN
4,0,249,NaN,0.000100,NaN,NaN,NaN
...,...,...,...,...,...,...,...
879,4,6349,NaN,0.000086,NaN,NaN,NaN
880,4,6399,NaN,0.000086,NaN,NaN,NaN
881,4,6449,NaN,0.000086,NaN,NaN,NaN
882,4,6499,NaN,0.000085,NaN,NaN,NaN
